In [ ]:
# ============================================
# TALLER 3 - FLUJO ANALITICO REPRODUCIBLE
# GRUPO 7
# ============================================

import pandas as pd
from pathlib import Path
from datetime import datetime
import os
import json
import requests
from IPython.display import display, HTML

print("="*60)
print("TALLER 3 - FLUJO ANALITICO REPRODUCIBLE")
print("GRUPO 7:")
print("- LESLY JAZMIN AMAGUA BARRIONUEVO")
print("- WILLIAM LEONARDO GRANDA GUERRERO")
print("- JORGE ENRIQUE ORDOÑEZ GARCIA")
print("- SAUL STALIN SALAZAR JARAMILLO")
print("="*60)

# ============================================
# 0. FUNCIONES REUTILIZABLES
# ============================================

def leer_csv_seguro(ruta, columnas_texto=None, columnas_numericas=None):
    """
    Lee un CSV con parametros estandarizados
    columnas_texto: lista de columnas que deben leerse como texto
    columnas_numericas: lista de columnas que deben leerse como numerico
    """
    dtype = {}
    if columnas_texto:
        for col in columnas_texto:
            dtype[col] = str
    if columnas_numericas:
        for col in columnas_numericas:
            dtype[col] = float

    df = pd.read_csv(
        ruta,
        dtype=dtype,
        na_values=['', 'NA', 'N/A'],
        encoding='utf-8'
    )

    # Asegurar que las columnas numericas realmente sean numericas
    if columnas_numericas:
        for col in columnas_numericas:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

def descargar_archivo(url, destino):
    """Descarga un archivo desde una URL"""
    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            with open(destino, 'wb') as f:
                f.write(response.content)
            return True
        return False
    except Exception as e:
        print(f"    Error: {e}")
        return False

# ============================================
# 1. CREAR ESTRUCTURA DE CARPETAS
# ============================================
print("\n[1] Creando estructura de carpetas del proyecto...")

BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
NOTEBOOKS_DIR = BASE_DIR / "notebooks"
SRC_DIR = BASE_DIR / "src"
REPORTS_DIR = BASE_DIR / "reports"
FIGURES_DIR = BASE_DIR / "figures"
DOCS_DIR = BASE_DIR / "docs"

for directorio in [RAW_DIR, PROCESSED_DIR, NOTEBOOKS_DIR, SRC_DIR,
                   REPORTS_DIR, FIGURES_DIR, DOCS_DIR]:
    directorio.mkdir(parents=True, exist_ok=True)

print("    Carpetas creadas correctamente")
print(f"    - {RAW_DIR}/")
print(f"    - {PROCESSED_DIR}/")
print(f"    - {NOTEBOOKS_DIR}/")
print(f"    - {SRC_DIR}/")
print(f"    - {REPORTS_DIR}/")
print(f"    - {FIGURES_DIR}/")
print(f"    - {DOCS_DIR}/")

# ============================================
# 2. DESCARGAR ARCHIVOS DESDE DRIVE
# ============================================
print("\n[2] Descargando archivos desde Drive...")

# IDs de los archivos en la carpeta compartida
ID_ESTUDIANTES = "1DQDKhrKlwlh_G1zMSiddQUu-NJzv7beW"
ID_MATRIZ = "1P1Y5WPnI28IOim7FtBwS7WcRkKjjI7l8"
ID_EXTENSION = "1UtFx-pxN9aabJF9CVcXvP8AW_qYlviVp"

# Descargar estudiantes
print("    Descargando estudiantes_master.xlsx...")
ruta_estudiantes = RAW_DIR / 'estudiantes_master.xlsx'
url_estudiantes = f"https://drive.google.com/uc?export=download&id={ID_ESTUDIANTES}"
if descargar_archivo(url_estudiantes, ruta_estudiantes):
    print("     estudiantes_master.xlsx descargado")
else:
    print("     Error al descargar estudiantes_master.xlsx")

# Descargar entregas matriz
print("    Descargando entregas_campus_matriz.csv...")
ruta_matriz = RAW_DIR / 'entregas_campus_matriz.csv'
url_matriz = f"https://drive.google.com/uc?export=download&id={ID_MATRIZ}"
if descargar_archivo(url_matriz, ruta_matriz):
    print("     entregas_campus_matriz.csv descargado")
else:
    print("     Error al descargar entregas_campus_matriz.csv")

# Descargar entregas extension
print("    Descargando entregas_campus_extension.csv...")
ruta_extension = RAW_DIR / 'entregas_campus_extension.csv'
url_extension = f"https://drive.google.com/uc?export=download&id={ID_EXTENSION}"
if descargar_archivo(url_extension, ruta_extension):
    print("     entregas_campus_extension.csv descargado")
else:
    print("     Error al descargar entregas_campus_extension.csv")

# Verificar archivos descargados
print("\n    Archivos en data/raw:")
for f in os.listdir(RAW_DIR):
    tamaño = os.path.getsize(RAW_DIR / f)
    print(f"    - {f} ({tamaño} bytes)")

# ============================================
# 3. LECTURA DE ARCHIVOS
# ============================================
print("\n[3] Leyendo archivos...")

# Leer estudiantes (Excel)
try:
    df_estudiantes = pd.read_excel(
        ruta_estudiantes,
        dtype={'id_estudiante': str},
        engine='openpyxl'
    )
    print(f"    estudiantes_master.xlsx: {len(df_estudiantes)} registros (Excel)")
except Exception as e:
    print(f"    Error con Excel: {e}")
    try:
        df_estudiantes = pd.read_csv(
            ruta_estudiantes,
            dtype={'id_estudiante': str},
            encoding='utf-8'
        )
        print(f"    estudiantes_master.xlsx: {len(df_estudiantes)} registros (CSV)")
    except Exception as e2:
        print(f"    Error con CSV: {e2}")
        df_estudiantes = pd.DataFrame()

# Leer entregas usando la funcion
archivos = {
    'matriz': ruta_matriz,
    'extension': ruta_extension
}

dataframes = {}
for nombre, ruta in archivos.items():
    dataframes[nombre] = leer_csv_seguro(
        ruta,
        columnas_texto=['id_estudiante'],
        columnas_numericas=['puntaje_obtenido']
    )
    print(f"    entregas_{nombre}: {len(dataframes[nombre])} registros")

# Asignar DataFrames
df_matriz = dataframes['matriz']
df_extension = dataframes['extension']

# ============================================
# 4. VALIDACION DE DATOS
# ============================================
print("\n[4] Validando datos de entrada...")

if not df_estudiantes.empty and not df_matriz.empty and not df_extension.empty:
    ids_estudiantes = set(df_estudiantes['id_estudiante'])
    ids_matriz = set(df_matriz['id_estudiante'])
    ids_extension = set(df_extension['id_estudiante'])

    ids_huerfanos = (ids_matriz.union(ids_extension)) - ids_estudiantes

    print(f"    IDs en catalogo: {len(ids_estudiantes)}")
    print(f"    IDs en entregas matriz: {len(ids_matriz)}")
    print(f"    IDs en entregas extension: {len(ids_extension)}")

    if ids_huerfanos:
        print(f"     IDs huerfanos encontrados: {sorted(ids_huerfanos)}")
    else:
        print("    No se encontraron IDs huerfanos")

    # Puntajes nulos
    nulos_matriz = df_matriz['puntaje_obtenido'].isna().sum()
    nulos_extension = df_extension['puntaje_obtenido'].isna().sum()
    print(f"    Puntajes nulos en matriz: {nulos_matriz}")
    print(f"    Puntajes nulos en extension: {nulos_extension}")

    # Duplicados en catalogo
    duplicados = df_estudiantes['id_estudiante'].duplicated().sum()
    print(f"    Duplicados en catalogo: {duplicados}")

    if duplicados > 0:
        print("    ADVERTENCIA: Eliminando estudiantes duplicados...")
        df_estudiantes = df_estudiantes.drop_duplicates(subset=['id_estudiante'])
        print(f"    Registros despues de eliminar duplicados: {len(df_estudiantes)}")
else:
    print("    ERROR: Uno o mas archivos estan vacios o no se pudieron leer")




TALLER 3 - FLUJO ANALITICO REPRODUCIBLE
GRUPO 7:
- LESLY JAZMIN AMAGUA BARRIONUEVO
- WILLIAM LEONARDO GRANDA GUERRERO
- JORGE ENRIQUE ORDOÑEZ GARCIA
- SAUL STALIN SALAZAR JARAMILLO

[1] Creando estructura de carpetas del proyecto...
    Carpetas creadas correctamente
    - /content/data/raw/
    - /content/data/processed/
    - /content/notebooks/
    - /content/src/
    - /content/reports/
    - /content/figures/
    - /content/docs/

[2] Descargando archivos desde Drive...
    Descargando estudiantes_master.xlsx...
     estudiantes_master.xlsx descargado
    Descargando entregas_campus_matriz.csv...
     entregas_campus_matriz.csv descargado
    Descargando entregas_campus_extension.csv...
     entregas_campus_extension.csv descargado

    Archivos en data/raw:
    - estudiantes_master.xlsx (4756 bytes)
    - entregas_campus_matriz.csv (10497 bytes)
    - entregas_campus_extension.csv (10506 bytes)

[3] Leyendo archivos...
    Error con Excel: File is not a zip file
    estudiantes_m

,carrera,estado_entrega,total_entregas,puntaje_promedio,puntaje_min,puntaje_max
0,Ciencias de Datos,Aprobado,119,92.80,80.0,100.0
1,Ciencias de Datos,Pendiente,2,NaN,NaN,NaN
2,Ingeniería de Software,Aprobado,154,90.13,76.0,100.0
3,Ingeniería de Software,Pendiente,6,NaN,NaN,NaN
4,Ingeniería de Software,Revisión,14,73.43,65.0,79.0



     Resumen por campus:


,campus_origen_entrega,total_entregas,puntaje_promedio
0,Extension,150,90.70
1,Matriz,150,89.99



     Resumen por materia:


...


     RESUMEN DEL DATASET FINAL:
    - Registros totales: 300
    - Columnas totales: 15
    - Memoria utilizada: 0.25 MB

     DISTRIBUCION DE ESTADOS DE ENTREGA:


,Cantidad,count
0,Aprobado,276
1,Revisión,15
2,Pendiente,9



     DISTRIBUCION POR CARRERA:


,Cantidad,count
0,Ingeniería de Software,174
1,Ciencias de Datos,121



FLUJO ANALITICO COMPLETADO 

RESUMEN FINAL:
- Entregas procesadas: 300
- Entregas sin estudiante: 5
- Puntajes nulos: 9
- Archivo final: entregas_consolidadas_gold_20260816.csv


In [ ]:
df_final.head()

,id_entrega,id_estudiante,fecha_subida,materia,tipo_proyecto,puntaje_obtenido,estado_entrega,campus_origen_entrega,nombre_completo,fecha_nacimiento,carrera,campus_origen,tipo_beca,estado_academico,fecha_procesamiento
0,PRJ-1001,E001,2026-08-01,Sistemas de Bases de Datos,Modelado ER,95.5,Aprobado,Matriz,Mateo Alvarado,1998-05-12,Ingeniería de Software,Matriz,Completa,Activo,2026-08-16
1,PRJ-1002,E002,2026-08-02,Desarrollo de Software,Prototipo,88.0,Aprobado,Matriz,Camila Sánchez,2000-02-20,Ingeniería de Software,Matriz,Parcial,Activo,2026-08-16
2,PRJ-1003,E003,2026-08-03,DataOps,Pipeline,92.0,Aprobado,Matriz,Santiago Ríos,1999-03-10,Ciencias de Datos,Matriz,Ninguna,Activo,2026-08-16
3,PRJ-1004,E006,2026-08-04,Sistemas de Bases de Datos,Script SQL,100.0,Aprobado,Matriz,Martina Vega,2000-06-18,Ingeniería de Software,Matriz,Completa,Activo,2026-08-16
4,PRJ-1005,E001,2026-08-05,Desarrollo de Software,API REST,85.0,Aprobado,Matriz,Mateo Alvarado,1998-05-12,Ingeniería de Software,Matriz,Completa,Activo,2026-08-16
